# Data Quality, Cleaning and Exploratory Analysis
## From a Raw Source to a Reproducible Analytical Table

**Laboratory 1 — individual work — maximum 10 points**

Practical 4 gave you a guided pipeline:

$$
\text{raw table}
\rightarrow
\text{inspect}
\rightarrow
\text{type}
\rightarrow
\text{validate}
\rightarrow
\text{clean}
\rightarrow
\text{transform}
\rightarrow
\text{aggregate}
\rightarrow
\text{visualize}
$$

This laboratory uses a **different, real dataset** and removes most of the guidance.

The question is no longer “which pandas method should I call?” It is:

> Can you make defensible decisions about an unfamiliar dataset, encode those decisions in a
> reproducible pipeline, and communicate only what the data actually support?

### Learning outcomes

After this laboratory you should be able to:

1. describe the provenance and unit of analysis of a dataset
2. distinguish storage representation from variable meaning
3. audit missing, malformed, duplicated and semantically questionable values
4. make and document cleaning decisions without silently destroying information
5. express table transformations with vectorized pandas operations
6. produce grouped summaries at the level required by an analytical question
7. use visualizations for both exploration and data-quality checks
8. turn exploratory work into a reproducible cleaning pipeline
9. state conclusions without confusing association with causation

### Requirements

Use `pandas`, `numpy` and `matplotlib`. Additional libraries are not required.

Your finished notebook must run from a **fresh kernel** using **Run All**.

### Laboratory map

| Part | What you need to demonstrate |
|---|---|
| 0. Source | you know where the data came from and what one record means |
| 1. Audit | you inspect before changing anything |
| 2. Representation | missingness, dtypes and validity are explicit decisions |
| 3. Pipeline | trusted transformations become a reproducible function |
| 4. Features | derived quantities have a stated meaning |
| 5. EDA | aggregation and figures answer questions rather than demonstrate APIs |
| 6. Contract | assumptions become executable checks |
| 7. Report | conclusions and limitations are traceable to evidence |

### Submission contract

Submit one notebook named:

```text
Lab 01 - <Surname> <Name>.ipynb
```

Rules:

- do not edit or overwrite the raw source file
- do not hard-code final tables or numerical answers
- do not use `iterrows()` / `itertuples()` or row-by-row Python loops for data cleaning
- do not use `df.apply(..., axis=1)` merely as a substitute for vectorized operations
- do not use a global `dropna()` as a cleaning strategy
- do not replace missing values with `0` unless zero has a documented semantic meaning
- do not silently “correct” a category or value merely because it looks unusual
- keep the original `raw` DataFrame unchanged
- every figure must have a readable title and labeled axes; include units when they are known
- every conclusion must be traceable to code and output in the notebook

A **question-specific** complete-case subset such as
`df.dropna(subset=["horsepower", "price"])` is allowed when the analysis genuinely requires those
variables. That is different from deleting every row that contains any missing value.

---
## 0. Dataset and provenance

We use the **Automobile** dataset from the UCI Machine Learning Repository.

The source documentation describes three kinds of information in each record:

1. automobile specifications
2. an assigned insurance-risk rating
3. normalized insurance losses

The dataset is based on sources including the **1985 Ward's Automotive Yearbook**.

- [UCI dataset page](https://archive.ics.uci.edu/dataset/10/automobile)
- [Original raw data](https://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data)
- [Original source documentation](https://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.names)
- [DOI: 10.24432/C5B01C](https://doi.org/10.24432/C5B01C)
- License: **CC BY 4.0**

The source contains **205 records** and uses `?` as a missing-value token.

The raw file has no header. The column names below come from the source documentation.
Adding metadata is not the same thing as cleaning the observations.

### Important

This is a historical dataset with an incompletely documented sampling process. It can describe the
records that were collected; broader generalization requires assumptions that the source does not
fully establish.

The course repository should normally include the source snapshot in `data/`. The download fallback
below exists only so the notebook can recover when the file is missing.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data"
NAMES_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.names"

RAW_PATH = DATA_DIR / "imports-85.data"
NAMES_PATH = DATA_DIR / "imports-85.names"

for url, path in [(DATA_URL, RAW_PATH), (NAMES_URL, NAMES_PATH)]:
    if not path.exists():
        print(f"{path} not found locally; downloading the UCI source snapshot...")
        try:
            urlretrieve(url, path)
        except Exception as exc:
            raise RuntimeError(
                f"Could not download {path.name}. Download it manually from {url} "
                f"and place it in {path}."
            ) from exc

print("Raw data:      ", RAW_PATH)
print("Documentation: ", NAMES_PATH)

In [ ]:
COLUMNS = [
    "symboling",
    "normalized_losses",
    "make",
    "fuel_type",
    "aspiration",
    "num_doors",
    "body_style",
    "drive_wheels",
    "engine_location",
    "wheel_base",
    "length",
    "width",
    "height",
    "curb_weight",
    "engine_type",
    "num_cylinders",
    "engine_size",
    "fuel_system",
    "bore",
    "stroke",
    "compression_ratio",
    "horsepower",
    "peak_rpm",
    "city_mpg",
    "highway_mpg",
    "price",
]

raw = pd.read_csv(
    RAW_PATH,
    header=None,
    names=COLUMNS,
)

raw.head()

### Compact data dictionary

Read the original `imports-85.names` documentation when you need the exact source definition of a
field.

| Group | Columns | Interpretation |
|---|---|---|
| Insurance / risk | `symboling`, `normalized_losses` | risk rating and normalized losses |
| Identity / category | `make`, `fuel_type`, `aspiration`, `body_style`, `drive_wheels`, `engine_location` | categorical vehicle characteristics |
| Count-like representation | `num_doors`, `num_cylinders` | recorded as words |
| Geometry / mass | `wheel_base`, `length`, `width`, `height`, `curb_weight` | vehicle dimensions / weight |
| Engine | `engine_type`, `engine_size`, `fuel_system`, `bore`, `stroke`, `compression_ratio` | engine characteristics |
| Performance | `horsepower`, `peak_rpm` | engine performance |
| Efficiency | `city_mpg`, `highway_mpg` | fuel economy in miles per gallon |
| Price | `price` | recorded vehicle price |

**Do not treat every min/max printed in the source documentation as a validity rule.** Some ranges
describe what happens to be observed in this dataset. A domain invariant needs a semantic reason,
not merely an observed minimum and maximum.

---
## 1. First-pass audit — understand before cleaning

Start from `raw`.

Your first task is **not** to repair anything.

### 1.1 What does this dataset actually describe?

Use inspection code and answer in Markdown:

1. What does one row appear to represent?
2. Is there an explicit unique identifier?
3. What can this dataset describe directly?
4. What broader population might someone *want* to generalize to, and what information is missing
   from the source to justify that generalization?
5. Which columns appear numerical, categorical, ordinal or count-like?

Do not invent sampling information that the source does not provide.

In [ ]:
# Your solution starts here.
# Inspect shape, dtypes, representative rows, summaries and category values.

### 1.2 Build an audit table

Construct one DataFrame with **one row per source column** and at least:

- `storage_dtype`
- `n_missing_inferred`
- `n_source_missing_token`
- `missing_or_token_pct`
- `n_unique_non_missing`

Add at least **one additional diagnostic** that you consider useful.

Sort the table so the columns with the largest apparent data-quality burden appear first.

The purpose is to make one fact visible:

> a value can be missing in the source even when pandas did not infer it as `NaN`.

Do not clean the dataset to build this table.

In [ ]:
# Your solution starts here.

### 1.3 Duplicates are a semantic question

Check for exact duplicate rows.

Then choose a **plausible subset of specification columns** and investigate whether several records
share the same values on that subset while differing elsewhere, for example in price or insurance
information.

Explain:

> Would any of these observations justify automatically deleting a row?

A repeated specification is not automatically a duplicate entity.

In [ ]:
# Your solution starts here.

---
## 2. Missingness, representation and validity

The source uses `?` for missing values.

A parser can replace `?` with `NaN`; that mechanical step is easy. The analytical question is what
becomes missing, whether that missingness is concentrated in particular groups, and which analyses
are affected.

### 2.1 Make source missingness explicit

Create a working copy and convert the source missing-value token to a proper pandas missing value.

Then show:

- missing count per column
- missing percentage per column
- one grouped missingness diagnostic

For the grouped diagnostic, compare the observed missingness rate of one important variable across a
meaningful category such as `make`, `body_style`, or another grouping you justify.

You may describe an observed pattern. You **cannot prove MCAR, MAR or MNAR from this table alone**.

In [ ]:
# Your solution starts here.

### 2.2 Convert dtypes deliberately

Identify fields that are conceptually numerical but were not loaded as numerical values.

Convert them explicitly.

Requirements:

- use `pd.to_numeric(..., errors="coerce")` or an equivalently explicit approach
- first check which non-missing tokens you are about to coerce
- preserve categorical meaning for categorical variables
- preserve missing values unless a later analysis has a justified imputation rule
- show dtypes before and after conversion

You may use `convert_dtypes()` as a starting point, as in Practical 4, but it does **not** replace
explicit numerical parsing.

In [ ]:
# Your solution starts here.

### 2.3 Count-like text representations

`num_doors` and `num_cylinders` are recorded as words.

Create numerical companion columns:

```text
num_doors_n
num_cylinders_n
```

Missing or unknown values must remain missing.

Keep the original source columns as well. The numerical columns are an analytical representation,
not a correction of the raw source.

In [ ]:
# Your solution starts here.

### 2.4 Define data-quality rules

Define at least **four executable checks** based on the source definition and the meaning of the
variables.

At least:

- one check must concern a documented categorical / ordinal domain
- one must concern a quantity that has a semantic positivity or non-negativity requirement
- one must concern one of your count-like numerical representations
- one must be an additional check of your choice

Distinguish:

```text
missing value
≠
invalid value
≠
unusual but valid value
```

Do not turn an observed min/max into an invariant merely because it appears in the documentation.

In [ ]:
# Your solution starts here.

---
## 3. From exploratory cells to a reproducible cleaning pipeline

Implement:

```python
def clean_automobile(raw: pd.DataFrame) -> pd.DataFrame:
    ...
```

The function must:

1. leave `raw` unchanged
2. make the source missing-value representation explicit
3. convert numerical variables to appropriate numerical dtypes
4. create `num_doors_n` and `num_cylinders_n`
5. apply only rules that you can justify
6. preserve legitimate missing values
7. return a new DataFrame

The target is **not** a table with zero missing values.

The target is a table whose representation and assumptions are explicit enough that the next analysis
step has a defensible meaning.

In [ ]:
def clean_automobile(raw: pd.DataFrame) -> pd.DataFrame:
    # Your solution starts here.
    raise NotImplementedError

In [ ]:
# Run your function and make the claims executable:
#
# 1. raw was not modified
# 2. the cleaned table has the intended dtypes
# 3. your data-quality checks pass
#
# Hint: pd.testing.assert_frame_equal(...) is useful for checking that raw stayed unchanged.

---
## 4. Derived variables: create quantities with a meaning

Create at least these two features.

### 4.1 Power-to-weight ratio

```text
power_to_weight = horsepower / curb_weight × 1000
```

Interpret it as recorded horsepower per 1000 units of recorded curb weight.

### 4.2 Highway–city MPG gap

```text
mpg_gap = highway_mpg - city_mpg
```

Then create **one additional derived variable of your own**.

For every derived variable, state:

- what it represents
- when it is undefined because an input is missing
- why it may be useful for an analytical question

Do not fill a missing input merely to force the formula to produce a number.

In [ ]:
# Your solution starts here.

---
## 5. Exploratory analysis — answer questions, not APIs

Answer **Questions A–C** and formulate **Question D** yourself.

For each question:

1. create a clearly named result table when aggregation is involved
2. report at least one numerical summary
3. interpret the result in Markdown
4. state any missing-data or sample-size limitation that materially affects the answer

Avoid causal language. This dataset can show associations; it does not establish why they occur.

### Question A — Price structure

How does observed vehicle price vary across **body styles**?

Include:

- the number of automobiles with usable price in every body style
- median price
- at least one measure of spread or distribution shape
- a discussion of whether small groups deserve the same interpretive weight as large groups
- an appropriate visualization

In [ ]:
# Your solution starts here.

### Question B — Performance and price

Investigate the relationship between **horsepower**, **curb weight** and **price**.

Use `power_to_weight` somewhere in the analysis.

Include:

- a question-specific complete-case subset
- at least one numerical association summary
- one visualization of a numerical relationship
- a brief justification of the association statistic you chose
- one sentence explaining what the figure or statistic does **not** prove

In [ ]:
# Your solution starts here.

### Question C — Fuel efficiency

Compare city and highway fuel efficiency across one meaningful grouping.

You may use, for example:

- `fuel_type`
- `drive_wheels`
- `aspiration`
- `body_style`

or another grouping you justify.

Use `mpg_gap`, and always report group sizes alongside the summary statistic.

In [ ]:
# Your solution starts here.

### Question D — Your own analytical question

Formulate one question that is **not** a trivial restatement of A–C.

Requirements:

- use at least two variables
- include either grouping or a relationship between numerical variables
- produce one compact result table or visualization
- explain why the aggregation level matches the question

Write the question first, then answer it.

In [ ]:
# Your solution starts here.

### Visualization contract

Across Questions A–D, include at least **three purposeful figures** serving different analytical roles:

1. a distribution or group-comparison figure
2. a relationship figure for numerical variables
3. one additional figure chosen for your question

Each figure must have:

- a readable title
- labeled axes
- units when the source makes them known
- no unexplained visual encoding
- a short interpretation directly below the figure

A figure is not automatically evidence of a meaningful pattern. Check sample size and missingness.

---
## 6. Final analytical table and a small contract

Create a final DataFrame named:

```python
analysis_df
```

It should contain the cleaned and derived variables you actually use and trust for this laboratory.

Then turn assumptions into executable checks.

Your contract must cover at least:

- expected columns
- numerical dtype for key analytical variables
- at least two domain invariants
- deliberate preservation of missing values where missingness remains legitimate

Do **not** assert that every column is complete.

In [ ]:
# Your solution starts here.

---
## 7. Reproducibility — rerun the whole chain

Your final notebook should reproduce this chain without manual intervention:

```text
imports-85.data
      ↓
pd.read_csv(...)
      ↓
clean_automobile(...)
      ↓
derived variables
      ↓
analysis_df
      ↓
tables + figures + conclusions
```

Save the final analytical table to:

```text
data/automobile_clean.csv
```

Do not overwrite the raw source file. Do not save the DataFrame index unless you intentionally made
it part of the data model.

In [ ]:
# Your solution starts here.

---
## 8. Final report

Finish the notebook with a concise Markdown report of approximately **200–350 words**.

### Data-quality findings

Describe at least two important issues in the source data and what you did about them.

### Analytical findings

State at least three findings that are directly supported by your tables or figures.

### Limitations

State at least two important limitations. At least one must concern **source / coverage / sampling**,
not merely the number of rows.

### One decision you deliberately rejected

Give one tempting cleaning or analytical step that you chose **not** to perform because it would
require an unjustified assumption.

---
## Grading rubric — 10 points

| Criterion | Points | Full-credit evidence |
|---|---:|---|
| **1. Provenance and first-pass audit** | **1.5** | Correct unit-of-analysis reasoning, useful audit table, source limitations, duplicate reasoning |
| **2. Missingness, representation and cleaning decisions** | **2.5** | Source missingness handled explicitly, dtypes match semantics, decisions justified, no destructive blanket cleaning |
| **3. Reproducible pipeline and data contract** | **1.5** | `clean_automobile()` is reproducible, does not mutate `raw`, analytical representations are consistent, checks encode real invariants |
| **4. Exploratory analysis** | **2.0** | A–D are answered correctly, aggregation matches each question, missingness/sample size are considered, claims remain observational |
| **5. Visualizations** | **1.5** | At least three purposeful and readable figures with correct labels, known units and interpretation |
| **6. Final report and reproducibility** | **1.0** | Fresh-kernel execution succeeds, final table is exported correctly, findings and limitations are traceable to results |
| **Total** | **10.0** | |

### What does not earn extra credit

- using more libraries
- producing many plots without a question
- removing every missing value
- writing the shortest possible code
- using a sophisticated method you cannot explain

The quality of the reasoning matters more than the number of API calls.

---
## Submission checklist

- [ ] `Restart Kernel` → `Run All` succeeds
- [ ] the raw file was not edited or overwritten
- [ ] `raw` remains unchanged by `clean_automobile()`
- [ ] source missingness and cleaning decisions are documented
- [ ] no row-wise cleaning loops are used
- [ ] no blanket `dropna()` / `fillna(0)` is used without a domain reason
- [ ] Questions A–D are answered
- [ ] at least three purposeful figures are present
- [ ] group summaries report sample counts where they matter
- [ ] `analysis_df` has executable quality checks
- [ ] `data/automobile_clean.csv` is generated by code
- [ ] conclusions distinguish observation from explanation

### Further reading

- [UCI Automobile dataset](https://archive.ics.uci.edu/dataset/10/automobile)
- [pandas: Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [pandas: Indexing and selecting data](https://pandas.pydata.org/docs/user_guide/indexing.html)
- [pandas: GroupBy](https://pandas.pydata.org/docs/user_guide/groupby.html)
- [pandas: Chart visualization](https://pandas.pydata.org/docs/user_guide/visualization.html)

The closing rule from Practical 4 still applies:

> **A clean DataFrame is not one with no `NaN`s. It is one whose representation and assumptions are
> explicit enough that the next analysis step has a defensible meaning.**